In [1]:
import jieba
from gensim import corpora, models
from docx import Document

In [3]:
def read_docx(file_path):
    # 打开并读取 .docx 文件
    doc = Document(file_path)

    # 提取文档中的所有段落
    doc_text = []
    for para in doc.paragraphs:
        doc_text.append(para.text)
    
    # 将所有段落合并为单个字符串，并按换行符分隔
    full_text = '，'.join(doc_text)
    
    return full_text

path = '人民日报文档.docx'
path_2 = '人民日报文档2.docx'
path_4 = '人民日报文档4.docx'
documents = [read_docx(path),read_docx(path_2)]
read_docx(path),read_docx(path_2)

('新华社北京10月15日电（记者宋晨 张泉）国家航天局系统工程司司长杨小宇在15日举行的国新办发布会上介绍，我国未来在月球探测方面，将发射嫦娥七号、嫦娥八号。嫦娥七号要对月球南极环境和资源进行探测，嫦娥八号将开展月球资源就位利用的技术验证。，“嫦娥七号和嫦娥八号会构成正在论证的月球科研站基本型。”杨小宇说，两者还会联合对月球内部结构进行多物理场的综合探测。正在论证的国际月球科研站将持续开展科学探测研究、资源开发利用，包括一些前沿技术验证，是多学科、多目标、大规模的科技活动。，杨小宇表示，行星探测方面，将发射天问二号、天问三号、天问四号。天问二号将对小行星进行采样返回，首先对小行星进行环绕综合探测，然后采样返回，对小行星演化和太阳系早期历史进行研究；天问三号将进行火星采样返回，对火星环境进行探测；天问四号将对木星和木星的卫星进行研究，对木星空间和内部结构进行探测。，我国还将实施载人月球探测工程。中国载人航天工程新闻发言人、中国载人航天工程办公室副主任林西强介绍，将发挥好航天员在月面开展探测活动的独特优势，为我国探索地外天体提供更为广阔的历史机遇。，“我们将统筹利用首次载人登月前的飞行试验以及载人登月的任务机会，开展较大规模的空间科学实验，目前我们初步规划了月球科学、月基科学和资源勘查利用3个领域9大方向科学目标。”林西强说。，嫦娥六号带回的月背样品研究进展如何？杨小宇介绍，目前，科学家正对样品进行整理，初步的物理、化学成分和结构的探测已完成，发现了大量信息，如月球早期演化和月球背面火山活动的信息，也包含了记录采样点火山活动历史的玄武岩，还包括来自其他区域的一些非玄武岩物质。下一步将按照月球样品分发有关政策，开展后续研究工作。，，',
 '据外媒报道，波兰近期获得美国30.8亿美元军事贷款，用于采购美国波音公司的“阿帕奇”攻击直升机。据悉，波兰今年8月与美国签署一项价值100亿美元的采购协议。根据协议，美国将在2028年至2032年向波兰交付96架“阿帕奇”攻击直升机。，据悉，此类大笔军购项目在波兰近年来的军购清单中并非首次。由于地区局势日益紧张，波兰持续提高军费支出，不断升级武器装备，尤其重视空中安全领域建设。，大规模采购军用飞机。2019年，波兰以46亿美元的价格从美国订购32架F-35战斗机。今年8月底，波兰接收首架F-35战斗机，成为东欧地区第一个拥有该型战斗

In [3]:
# documents = ["这是一个测试文档。", "另一个文档测试。", "这是关于机器学习的文档。"]
# 停用词表
stop_words = set(['的', '是', '在', '和', '了', '我'])  # 根据需要调整停用词

# 步骤 1：分词处理并去除停用词
texts = [[word for word in jieba.lcut(doc) if word not in stop_words] for doc in documents]

# 步骤 2：构建字典和语料库
dictionary = corpora.Dictionary(texts)

# 过滤极端频率的词
dictionary.filter_extremes(no_below=1, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in texts]

# 步骤 3：训练 LSI 和 LDA 模型
lsi_model = models.LsiModel(corpus=corpus, id2word=dictionary, num_topics=2)
lda_model = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=2)

# 步骤 4：关键词抽取
print("LSI 模型关键词：")
lsi_topics = lsi_model.show_topics(num_topics=2, num_words=5, formatted=False)
for idx, topic in lsi_topics:
    print(f"主题 {idx + 1}: {[word for word, weight in topic]}")

print("\nLDA 模型关键词：")
lda_topics = lda_model.show_topics(num_topics=2, num_words=5, formatted=False)
for idx, topic in lda_topics:
    print(f"主题 {idx + 1}: {[word for word, weight in topic]}")

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\25232\AppData\Local\Temp\jieba.cache
Loading model cost 0.449 seconds.
Prefix dict has been built successfully.


LSI 模型关键词：
主题 1: ['波兰', '年', '采购', '-', '导弹']
主题 2: ['月球', '嫦娥', '天问', '载人', '开展']

LDA 模型关键词：
主题 1: ['波兰', '月球', '嫦娥', '年', '导弹']
主题 2: ['波兰', '年', '采购', '-', '导弹']


In [4]:
import jieba.analyse

# 示例文本
text = read_docx(path)

# 使用 TextRank 提取关键词
keywords = jieba.analyse.textrank(text, topK=5, withWeight=True)
print("TextRank 提取的关键词：")
for word, weight in keywords:
    print(f"{word}: {weight}")


TextRank 提取的关键词：
探测: 1.0
月球: 0.9618883417933572
进行: 0.827404433074911
载人: 0.537427174950977
开展: 0.529125890132145


In [5]:
import jieba
import math
from collections import defaultdict

# 示例文档
doc = read_docx(path)

# 语料库
corpus = [
  read_docx(path_2)
]

# 步骤 1：分词并统计词频（TF）
words_doc = jieba.lcut(doc)
tf = defaultdict(int)
for word in words_doc:
    tf[word] += 1

# 计算词语总数
total_terms = len(words_doc)

# 计算 TF 值
tf = {word: count / total_terms for word, count in tf.items()}

# 步骤 2：计算逆文档频率（IDF）
idf = {}
total_docs = len(corpus) + 1  # 加上目标文档
all_docs = corpus + [doc]
for doc_text in all_docs:
    words = set(jieba.lcut(doc_text))
    for word in words:
        idf[word] = idf.get(word, 0) + 1

idf = {word: math.log(total_docs / (count)) for word, count in idf.items()}

# 步骤 3：计算 TF-IDF 值
tf_idf = {word: tf[word] * idf.get(word, 0) for word in tf}

# 步骤 4：关键词抽取
keywords = sorted(tf_idf.items(), key=lambda x: x[1], reverse=True)
print("TF-IDF 提取的关键词：")
for word, score in keywords[:5]:
    print(f"{word}: {score}")


TF-IDF 提取的关键词：
月球: 0.018240715277893296
嫦娥: 0.011607727904113917
天问: 0.009949481060669072
开展: 0.008291234217224226
载人: 0.008291234217224226
